In [ ]:
!apt-get install tesseract-ocr -qq
!pip install pytesseract opencv-python-headless jiwer -q

import pytesseract, cv2, time, csv
import numpy as np
from PIL import Image
from jiwer import cer
from datasets import load_dataset

ds = load_dataset("bsmock/pubtables-1m", split="train[:300]")

def preprocess(pil_img):
    img = np.array(pil_img.convert("L"))
    img = cv2.GaussianBlur(img, (3,3), 0)
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(img)

results = []
for i, sample in enumerate(ds):
    img = sample['image']
    gt  = sample.get('text', '')   # ground truth text if available

    # Raw OCR
    t0 = time.perf_counter()
    raw_text = pytesseract.image_to_string(img)
    t_raw = time.perf_counter() - t0

    # Preprocessed OCR
    t0 = time.perf_counter()
    proc_text = pytesseract.image_to_string(preprocess(img))
    t_proc = time.perf_counter() - t0

    results.append({
        'id': i,
        'raw_ocr_text': raw_text,
        'proc_ocr_text': proc_text,
        'raw_time': round(t_raw, 3),
        'proc_time': round(t_proc, 3),
        'ground_truth': gt
    })
    if i % 50 == 0: print(f"Done {i}/300")

# Save
import pandas as pd
pd.DataFrame(results).to_csv("results/ocr_results.csv", index=False)
print("Saved.")